In [ ]:
# IMPORTS

from __future__ import annotations
from transformers import AutoTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

C:\Users\Fcomm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class LLM:
    def __init__(self):
        model_name = "google/flan-t5-small"
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def ask(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt")
        outputs = self.model.generate(**inputs, max_length=50)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

# create instance
llm = LLM()

Loading tokenizer...


Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3682.24it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
def directional_agent(direction, state):
    prompt = f"""
Instruction: Choose the action {direction} and explain why it is the best move.

Environment description:
{state}

Think about:
- Is there a wall?
- Is the goal visible?
- Is the path clear?

Answer in one sentence explaining WHY {direction} is good.
"""
    return llm.ask(prompt)

In [4]:
def arbiter(state, forward_arg, left_arg, right_arg):
    prompt = f"""
Instruction: Decide the best action for the agent.

Environment:
{state}

Options:

FORWARD:
{forward_arg}

LEFT:
{left_arg}

RIGHT:
{right_arg}

Choose the best action based on the environment.

Answer ONLY in this format:
Action: FORWARD or LEFT or RIGHT
Reason: <short reason>
"""
    return llm.ask(prompt)

In [5]:
def multi_agent_step(state):
    forward = directional_agent("FORWARD", state)
    left = directional_agent("LEFT", state)
    right = directional_agent("RIGHT", state)

    print("FORWARD:", forward)
    print("LEFT:", left)
    print("RIGHT:", right)

    decision = arbiter(state, forward, left, right)

    return decision

In [6]:
state = "Front is empty. Left is a wall. Right is empty. Goal is ahead."

result = multi_agent_step(state)
print("FINAL:", result)

FORWARD: Is the path clear?
LEFT: Is the path clear?
RIGHT: Is the path clear?
FINAL: RIGHT


In [8]:
def local_agent(view_name, info):
    prompt = f"""
Environment:
- {info[0][0]}: {info[0][1]}
- {info[1][0]}: {info[1][1]}

Question: Which direction is blocked?

Answer with one word.
"""
    return llm.ask(prompt)

In [9]:
def get_views(state_dict):
    return {
        "left_front": [("Left", state_dict["Left"]), ("Front", state_dict["Front"])],
        "front_right": [("Front", state_dict["Front"]), ("Right", state_dict["Right"])],
        "left_right": [("Left", state_dict["Left"]), ("Right", state_dict["Right"])]
    }

In [10]:
def run_local_agents(state_dict):
    views = get_views(state_dict)
    outputs = {}

    for name, info in views.items():
        outputs[name] = local_agent(name, info)

    return outputs

In [11]:
def main_agent(state_dict, local_outputs):
    prompt = f"""
Environment:
- Left: {state_dict["Left"]}
- Front: {state_dict["Front"]}
- Right: {state_dict["Right"]}

Local observations:
- Left/Front agent says blocked: {local_outputs["left_front"]}
- Front/Right agent says blocked: {local_outputs["front_right"]}
- Left/Right agent says blocked: {local_outputs["left_right"]}

Question: Which direction should the agent move?

Rules:
- Do not choose a blocked direction
- Prefer open paths

Answer with one word: Left, Front, or Right.
"""
    return llm.ask(prompt)

In [12]:
def step(state_dict):
    local_outputs = run_local_agents(state_dict)

    print("Local outputs:", local_outputs)

    decision = main_agent(state_dict, local_outputs)

    return decision

In [16]:
state = {
    "Left": "wall",
    "Front": "empty",
    "Right": "wall"
}

result = step(state)
print("FINAL:", result)

Local outputs: {'left_front': 'Left', 'front_right': 'Right', 'left_right': 'Right'}
FINAL: Front


In [22]:
state = {"Left": "empty", "Front": "wall", "Right": "wall"}
result = step(state)
print("FINAL:", result)

Local outputs: {'left_front': 'Front', 'front_right': 'Right', 'left_right': 'Right'}
FINAL: Left


In [23]:
state = {"Left": "wall", "Front": "wall", "Right": "empty"}
result = step(state)
print("FINAL:", result)

Local outputs: {'left_front': 'Front', 'front_right': 'Right', 'left_right': 'Left'}
FINAL: Left
